# ex018_Hamann2015

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

CASE_DIR = Path.cwd()
OUTPUT_DIR = CASE_DIR / "output"
SIMULATION_DIR = CASE_DIR / "simulation"
import json

import flopy
import matplotlib as mpl
from scipy.interpolate import RectBivariateSpline

DAYS_PER_YEAR = 365.25
EARLY_YEARS = [1.0, 20.0, 40.0, 70.0]
LATE_YEARS = [1000.0, 2000.0, 3000.0, 4000.0, 5000.0, 6000.0]
NX_FINE, NZ_FINE = (700, 220)
SIM_DIR = SIMULATION_DIR
mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "font.family": "DejaVu Sans",
    }
)
metadata = json.loads((OUTPUT_DIR / "model_metadata.json").read_text(encoding="utf-8"))
results = np.load(OUTPUT_DIR / "results.npy")
headings = (OUTPUT_DIR / "results_headings.txt").read_text(encoding="utf-8-sig").splitlines()
years = np.load(OUTPUT_DIR / "result_times_years.npy")
delr = np.load(OUTPUT_DIR / "grid_delr_m.npy")
delv = np.load(OUTPUT_DIR / "grid_delv_m.npy")
nlay, ncol = (int(metadata["nlay"]), int(metadata["ncol"]))
x = np.cumsum(delr) - 0.5 * delr
z = 10.0 - (np.cumsum(delv) - 0.5 * delv)
x_fine = np.linspace(0.0, 100.0, NX_FINE)
z_fine = np.linspace(0.0, 10.0, NZ_FINE)
budget = flopy.utils.CellBudgetFile(str(SIM_DIR / "gwf_model.bud"), precision="double")


def year_index(year):
    matches = np.flatnonzero(np.isclose(years, year))
    return int(matches[0])


def interpolate_field(field, *, clip=False):
    spline = RectBivariateSpline(z[::-1], x, field[::-1], kx=2, ky=3, s=0.0)
    fine = spline(z_fine, x_fine)
    if clip:
        fine = np.clip(fine, np.nanmin(field), np.nanmax(field))
    return fine


def fields_at_year(year):
    idx = year_index(year)
    rho = results[idx, headings.index("RHO")].reshape(nlay, ncol) * 1000.0
    spdis = budget.get_data(text="DATA-SPDIS", totim=float(year * DAYS_PER_YEAR))[0]
    qx = np.asarray(spdis["qx"]).reshape(nlay, ncol)
    qz = np.asarray(spdis["qz"]).reshape(nlay, ncol)
    return (rho, qx, qz)


def add_flow_overlay(ax, qx, qz):
    qx_fine = interpolate_field(qx)
    qz_fine = interpolate_field(qz)
    speed = np.hypot(qx_fine, qz_fine)
    positive = speed[speed > 0.0]
    scale = np.nanpercentile(positive, 90) if positive.size else 1.0
    linewidth = 0.35 + 1.15 * np.clip(speed / max(scale, 1e-30), 0.0, 1.0)
    ax.streamplot(
        x_fine,
        z_fine,
        qx_fine,
        qz_fine,
        color="white",
        linewidth=linewidth,
        density=(1.45, 0.9),
        arrowsize=0.75,
        arrowstyle="-|>",
        minlength=0.08,
        integration_direction="both",
        broken_streamlines=True,
        zorder=4,
    )
    return float(np.nanmax(speed))


def style_cross_section(ax, year):
    ax.axvline(50.0, color="white", lw=1.0, ls="--", alpha=0.95, zorder=5)
    ax.text(49.2, 9.55, "recharge", color="white", ha="right", va="top", fontsize=8)
    ax.text(50.8, 9.55, "evaporation", color="white", ha="left", va="top", fontsize=8)
    ax.set_xlim(0.0, 100.0)
    ax.set_ylim(0.0, 10.0)
    ax.set_xlabel("Distance x (m)")
    ax.set_ylabel("Elevation z (m)")
    label = "year" if np.isclose(year, 1.0) else "years"
    ax.set_title(f"{year:g} {label}", loc="left", fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)


def density_panel(ax, year, *, vmin=None, vmax=None, levels=32):
    rho, qx, qz = fields_at_year(year)
    rho_fine = interpolate_field(rho, clip=True)
    if vmin is None:
        vmin = float(np.nanmin(rho))
    if vmax is None:
        vmax = float(np.nanmax(rho))
    if np.isclose(vmin, vmax):
        vmax = vmin + 0.01
    contour = ax.contourf(
        x_fine,
        z_fine,
        rho_fine,
        levels=np.linspace(vmin, vmax, levels),
        cmap="turbo",
        extend="both",
    )
    line_levels = np.linspace(vmin, vmax, 7)[1:-1]
    ax.contour(
        x_fine, z_fine, rho_fine, levels=line_levels, colors="k", linewidths=0.28, alpha=0.35
    )
    max_speed = add_flow_overlay(ax, qx, qz)
    style_cross_section(ax, year)
    ax.text(
        0.012,
        0.965,
        f"ρ = {rho.min():.2f}–{rho.max():.2f} kg m⁻³\n|max q| = {max_speed:.2e} m d⁻¹",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8,
        color="white",
        bbox=dict(facecolor="black", alpha=0.42, edgecolor="none", pad=2.5),
        zorder=6,
    )
    return (contour, rho)


fig, axes = plt.subplots(2, 2, figsize=(15.2, 6.6), constrained_layout=True)
for ax, year in zip(axes.flat, EARLY_YEARS, strict=False):
    contour, rho = density_panel(ax, year)
    cbar = fig.colorbar(contour, ax=ax, pad=0.015, fraction=0.035)
    cbar.set_label("Density (kg m⁻³)")
fig.suptitle("Early density instability: descending brine plumes", fontsize=15, fontweight="bold")
plt.show()

In [ ]:
late_density = [fields_at_year(year)[0] for year in LATE_YEARS]
late_vmin = float(min(np.nanmin(field) for field in late_density))
late_vmax = float(max(np.nanmax(field) for field in late_density))
fig, axes = plt.subplots(3, 2, figsize=(15.2, 9.2), constrained_layout=True)
last_contour = None
for ax, year in zip(axes.flat, LATE_YEARS, strict=False):
    last_contour, _ = density_panel(ax, year, vmin=late_vmin, vmax=late_vmax, levels=42)
cbar = fig.colorbar(last_contour, ax=axes, pad=0.012, fraction=0.024, shrink=0.92)
cbar.set_label("Density (kg m⁻³)")
fig.suptitle("Long-term density-driven circulation", fontsize=15, fontweight="bold")
plt.show()

In [ ]:
minerals = ("Calcite", "Gypsum", "Halite")
colors = plt.cm.viridis(np.linspace(0.04, 0.96, len(years)))
fig, axes = plt.subplots(3, 1, figsize=(11.5, 8.5), sharex=True, constrained_layout=True)
for ax, mineral in zip(axes, minerals, strict=False):
    mineral_idx = headings.index(mineral)
    for time_idx, (year, color) in enumerate(zip(years, colors, strict=False)):
        surface = results[time_idx, mineral_idx].reshape(nlay, ncol)[0]
        ax.plot(x, surface, color=color, lw=1.35, label=f"{year:g} y")
    ax.axvline(50.0, color="0.25", lw=0.9, ls="--")
    ax.set_ylabel(f"{mineral}\n(mol L⁻¹ bulk)")
    ax.grid(alpha=0.22)
    ax.set_xlim(0.0, 100.0)
axes[0].legend(ncol=6, fontsize=8, loc="upper left")
axes[-1].set_xlabel("Distance x (m)")
fig.suptitle("Surface mineral precipitation: bull's-eye transect", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
%config InlineBackend.figure_format = 'svg'
from matplotlib.ticker import MaxNLocator, NullLocator
from scipy.interpolate import RegularGridInterpolator

W = 183 / 25.4
mpl.rcdefaults()
mpl.rcParams.update(
    {
        "font.family": "Arial",
        "font.size": 9,
        "axes.labelsize": 9,
        "axes.titlesize": 9,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "legend.fontsize": 8.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.linewidth": 0.65,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "legend.frameon": False,
        "savefig.facecolor": "white",
    }
)
raw = np.load(OUTPUT_DIR / "results.npy")
headings = (OUTPUT_DIR / "results_headings.txt").read_text(encoding="utf-8-sig").splitlines()
years = np.load(OUTPUT_DIR / "result_times_years.npy")
dx = np.load(OUTPUT_DIR / "grid_delr_m.npy")
dz = np.load(OUTPUT_DIR / "grid_delv_m.npy")
shape = (len(dz), len(dx))
D = {
    "x_edges_m": np.r_[0, np.cumsum(dx)],
    "depth_edges_m": np.r_[0, np.cumsum(dz)],
    "time_yr": np.array([1, 20, 40, 70, 1000, 2000, 3000, 5000, 6000]),
}
with flopy.utils.CellBudgetFile(
    SIMULATION_DIR / "gwf_model.bud", precision="double"
) as paper_budget:
    for year in D["time_yr"]:
        idx = np.flatnonzero(np.isclose(years, year)).item()
        sp = paper_budget.get_data(text="DATA-SPDIS", totim=float(year) * 365.25)[0]
        D[f"density_{year}_kg_m3"] = raw[idx, headings.index("RHO")].reshape(shape) * 1000
        D[f"qx_{year}_m_d"] = sp["qx"].reshape(shape)
        D[f"qdown_{year}_m_d"] = -sp["qz"].reshape(shape)


def _manuscript_layout_06(fig):
    fig.canvas.draw()
    width_pt, height_pt = fig.get_size_inches() * 72
    offsets = [
        (0, 0),
        (0, 0),
        (0, 0),
        (0, -17.89343),
        (0, -17.89343),
        (0, -17.89343),
        (-0.83564532, -35.870325),
        (-0.83564532, -35.870325),
        (-0.83564532, -35.870325),
        (0, -3),
        (0, -3),
        (0, -3),
        (0, -20.89343),
        (0, -20.89343),
        (0, -20.89343),
        (-0.83564532, -38.870325),
        (-0.83564532, -38.870325),
        (-0.83564532, -38.870325),
    ]
    for ax, (dx, dy) in zip(fig.axes, offsets, strict=True):
        pos = ax.get_position()
        ax.set_position([pos.x0 + dx / width_pt, pos.y0 - dy / height_pt, pos.width, pos.height])
    for label in fig.texts:
        if label.get_text() in {"x (m)", "Distance (m)"}:
            x, y = label.get_position()
            label.set_position((x + -0.83564532 / width_pt, y - -38.870325 / height_pt))


def save(n, fig, axs):
    fig.canvas.draw()
    _manuscript_layout_06(fig)
    plt.show()
    plt.close(fig)


def title(ax, letter, text):
    ax.set_title(f"{letter}  {text}", loc="left", pad=7, fontsize=9, fontweight="normal")


def colorbar(fig, ax, im, unit, offset=0.048, height=0.01, ticks=None):
    pos = ax.get_position()
    ca = fig.add_axes([pos.x0, pos.y0 - offset, pos.width, height])
    cb = fig.colorbar(im, cax=ca, orientation="horizontal")
    cb.set_label(unit, fontsize=8.5, labelpad=2)
    cb.ax.tick_params(labelsize=8, pad=1, length=2)
    if ticks is None:
        cb.locator = MaxNLocator(3)
        cb.update_ticks()
    else:
        cb.set_ticks(ticks)
        cb.set_ticklabels([str(t) for t in ticks])
        cb.ax.xaxis.set_minor_locator(NullLocator())
    return cb


def fig6():
    d = D
    xe, ze = (d["x_edges_m"], d["depth_edges_m"])
    xc = (xe[1:] + xe[:-1]) / 2
    zc = (ze[1:] + ze[:-1]) / 2
    fig, axs = plt.subplots(3, 3, figsize=(W, 6.65))
    fig.subplots_adjust(left=0.074, right=0.98, top=0.92, bottom=0.135, wspace=0.27, hspace=1.08)
    fig.text(
        0.52,
        0.985,
        "0–50 m  Recharge      50–100 m  Evaporation",
        ha="center",
        va="top",
        fontsize=9,
    )
    xg = np.linspace(xc[0], xc[-1], 280)
    zg = np.linspace(zc[0], zc[-1], 100)
    X, Z = np.meshgrid(xg, zg)
    points = np.column_stack([Z.ravel(), X.ravel()])
    for i, (ax, t) in enumerate(zip(axs.flat, d["time_yr"], strict=False)):
        rho = d[f"density_{t}_kg_m3"]
        xp = np.r_[xe[0], xc, xe[-1]]
        zp = np.r_[ze[0], zc, ze[-1]]
        im = ax.contourf(
            xp,
            zp,
            np.pad(rho, 1, mode="edge"),
            levels=np.linspace(float(rho.min()), float(rho.max()), 25),
            cmap="turbo",
        )
        u = RegularGridInterpolator((zc, xc), d[f"qx_{t}_m_d"])(points).reshape(Z.shape)
        v = RegularGridInterpolator((zc, xc), d[f"qdown_{t}_m_d"])(points).reshape(Z.shape)
        ax.streamplot(xg, zg, u, v, color="white", density=0.7, linewidth=0.6, arrowsize=0.55)
        ax.set(xlim=(0, 100), ylim=(10, 0), xticks=[0, 50, 100], yticks=[0, 5, 10])
        ax.set_aspect(5)
        if i % 3 == 0:
            ax.set_ylabel("Depth (m)", labelpad=2)
        title(ax, chr(97 + i), f"{t:,} yr")
        cb = colorbar(fig, ax, im, "Density (kg/m³)", offset=0.058, height=0.012)
        cb.ax.xaxis.set_major_formatter(mpl.ticker.ScalarFormatter(useOffset=False))
        cb.locator = MaxNLocator(3)
        cb.update_ticks()
    fig.text(0.52, 0.012, "Distance (m)", ha="center", fontsize=9)
    save(6, fig, list(axs.flat))


fig6()

In [ ]:
%config InlineBackend.figure_format = 'svg'
W = 183 / 25.4
TIME4 = ["#477B92", "#6B9B88", "#C99659", "#986D91"]
mpl.rcdefaults()
mpl.rcParams.update(
    {
        "font.family": "Arial",
        "font.size": 9,
        "axes.titlesize": 9,
        "axes.labelsize": 8.5,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.linewidth": 0.65,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "savefig.facecolor": "white",
    }
)
raw = np.load(OUTPUT_DIR / "results.npy")
headings = (OUTPUT_DIR / "results_headings.txt").read_text(encoding="utf-8-sig").splitlines()
years = np.load(OUTPUT_DIR / "result_times_years.npy")
dx = np.load(OUTPUT_DIR / "grid_delr_m.npy")
dz = np.load(OUTPUT_DIR / "grid_delv_m.npy")
shape = (len(dz), len(dx))
D = {
    "x_m": np.cumsum(dx) - dx / 2,
    "averaging_thickness_m": np.array(0.5),
    "time_yr": np.array([2000, 4000, 5000, 6000]),
}
edges = np.r_[0, np.cumsum(dz)]
weights = np.maximum(0, np.minimum(edges[1:], 0.5) - edges[:-1])
idx = [np.flatnonzero(np.isclose(years, t)).item() for t in D["time_yr"]]
for mineral in ["Calcite", "Gypsum", "Halite"]:
    fields = raw[idx, headings.index(mineral)].reshape(-1, *shape)
    D[mineral] = np.einsum("tzx,z->tx", fields, weights) / weights.sum()


def _manuscript_layout_07(fig):
    fig.canvas.draw()
    width_pt, height_pt = fig.get_size_inches() * 72
    offsets = [(0, 0), (0, 0), (0, 0)]
    for ax, (dx, dy) in zip(fig.axes, offsets, strict=True):
        pos = ax.get_position()
        ax.set_position([pos.x0 + dx / width_pt, pos.y0 - dy / height_pt, pos.width, pos.height])
    for legend in fig.legends:
        box = legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted())
        legend.set_bbox_to_anchor(
            (box.x0 + 0 / width_pt, box.y0 - 3 / height_pt, box.width, box.height),
            transform=fig.transFigure,
        )


def save_section15(n, fig, axes):
    _manuscript_layout_07(fig)
    plt.show()


def title_section15(ax, letter, name):
    ax.set_title(f"{letter}  {name}", loc="left", pad=7, fontsize=9)


def grid(rows, cols, height, top=0.86, bottom=0.1, wspace=0.5, hspace=0.67):
    fig, axs = plt.subplots(rows, cols, figsize=(W, height), squeeze=False)
    fig.subplots_adjust(left=0.08, right=0.98, top=top, bottom=bottom, wspace=wspace, hspace=hspace)
    return (fig, list(axs.flat))


def fig7():
    d = D
    x = d["x_m"]
    fig, axs = grid(1, 3, 2.65, top=0.78, bottom=0.25, wspace=0.45)
    for i, (ax, name) in enumerate(zip(axs, ["Calcite", "Gypsum", "Halite"], strict=False)):
        for j, c in enumerate(TIME4):
            ax.plot(x, d[name][j], color=c, lw=1.5, label=f"{d['time_yr'][j]:,} yr")
        ax.set(xlim=(94, 100) if i == 2 else (0, 100), xlabel="Distance (m)", ylabel="mol/L bulk")
        ax.yaxis.set_major_locator(MaxNLocator(4))
        title_section15(ax, chr(97 + i), name)
    fig.legend(
        *axs[0].get_legend_handles_labels(), loc="upper center", ncol=4, bbox_to_anchor=(0.53, 1)
    )
    save_section15(7, fig, axs)


fig7()